# One synthetic applicant: load, bills and baseline model

Exploratory view of a single synthetic applicant produced by `reapschema.synth.generator`. All data shown are synthetic.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from reapschema import mv
from reapschema.config import load_config
from reapschema.provenance import REPO_ROOT
from reapschema.synth.generator import generate_applicant, spawn_seeds

cfg = load_config(REPO_ROOT / 'configs' / 'quick.yaml')
seeds = spawn_seeds(cfg.seed, 40)
app = next(a for a in (generate_applicant(i, seeds[i], cfg, error_class='none') for i in range(40)) if a.category == 'EEI')
app.state, app.facility_type, app.clean.technology, app.truth

## Billing records before and after installation

In [ ]:
ev = app.clean
fig, ax = plt.subplots(figsize=(7, 3))
for bills, label in ((ev.pre_bills, 'pre-installation'), (ev.post_bills, 'post-installation')):
    ax.step(bills.start, bills.printed / bills.days, where='post', label=label)
ax.axvline(ev.in_service_day, color='k', lw=0.8)
ax.set_xlabel('Day index'); ax.set_ylabel('Average daily use (kWh/day)'); ax.legend();

## Degree-day baseline and savings estimates

In [ ]:
table = mv.station_table(ev.state, cfg.mv.balance_grid_f)
est = mv.estimate_savings((ev.pre_bills.start, ev.pre_bills.end, ev.pre_bills.printed),
                           (ev.post_bills.start, ev.post_bills.end, ev.post_bills.printed),
                           table, cfg.mv.confidence)
{'undated difference': ev.pre_bills.printed.sum() - ev.post_bills.printed.sum(),
 'normalized annual savings': est.normalized, 'avoided energy use': est.avoided,
 'true normal-year savings': app.truth.savings_normal,
 'baseline CV(RMSE)': est.baseline.cvrmse, 'heating balance point (F)': est.baseline.t_h_f,
 'cooling balance point (F)': est.baseline.t_c_f, 'fractional savings uncertainty': est.fsu}